# Proyecto 3 — Análisis de Quejas Financieras Colombia (2015–2021)
**Fuente:** Superintendencia Financiera de Colombia  
**Dataset:** 3,754,916 registros originales · 8 variables  
**Herramientas:** Python · Pandas · Power BI  
**Autor:** Edison Andres Vega Rodriguez

In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
ruta = r"C:\Users\zerit\Downloads\QuejasSuper.csv"

# Cargar con low_memory=False para evitar errores de tipo en columnas mixtas
df = pd.read_csv(ruta, low_memory=False, encoding='utf-8')

print(f"Filas: {len(df):,}")
print(f"Columnas: {df.columns.tolist()}")
print(f"\nTipos de datos:\n{df.dtypes}")

Filas: 3,754,916
Columnas: ['TIPO_ENTIDAD', 'CODIGO_ENTIDAD', 'NOMBRE_ENTIDAD', 'PRODUCTO', 'MOTIVO', 'ESTADO', 'ANIO', 'MES']

Tipos de datos:
TIPO_ENTIDAD      object
CODIGO_ENTIDAD    object
NOMBRE_ENTIDAD    object
PRODUCTO          object
MOTIVO            object
ESTADO            object
ANIO              object
MES                int64
dtype: object


In [2]:
# Primeras filas
print(df.head(10))
print(f"\nShape: {df.shape}")

  TIPO_ENTIDAD CODIGO_ENTIDAD                        NOMBRE_ENTIDAD  \
0            1             57                  BANCO PICHINCHA S.A.   
1           23             10  COLFONDOS S.A. PENSIONES Y CESANTIAS   
2            1             49                             AV VILLAS   
3            1              7                           BANCOLOMBIA   
4            1             39                      BANCO DAVIVIENDA   
5            1              7                           BANCOLOMBIA   
6            1              7                           BANCOLOMBIA   
7            1             49                             AV VILLAS   
8            1             49                             AV VILLAS   
9           13              1                  ALLIANZ SEGUROS S.A.   

                           PRODUCTO  \
0  CREDITO DE CONSUMO Y/O COMERCIAL   
1                  PENSION DE VEJEZ   
2                               NaN   
3                TARJETA DE CREDITO   
4                  CUEN

In [3]:
#Analisis de calidad
print("=== NULOS POR COLUMNA ===")
print(df.isnull().sum())

print("\n=== VALORES ÚNICOS CLAVE ===")
print(f"ESTADO:       {df['ESTADO'].unique()}")
print(f"ANIO:         {sorted(df['ANIO'].dropna().unique())}")
print(f"TIPO_ENTIDAD únicos: {df['TIPO_ENTIDAD'].nunique()}")

print("\n=== TOP 10 ENTIDADES POR QUEJAS ===")
print(df['NOMBRE_ENTIDAD'].value_counts().head(10))

print("\n=== TOP 10 PRODUCTOS ===")
print(df['PRODUCTO'].value_counts().head(10))

print("\n=== TOP 10 MOTIVOS ===")
print(df['MOTIVO'].value_counts().head(10))

=== NULOS POR COLUMNA ===
TIPO_ENTIDAD        2565
CODIGO_ENTIDAD      2565
NOMBRE_ENTIDAD      2565
PRODUCTO          305140
MOTIVO            306251
ESTADO                 0
ANIO                   0
MES                    0
dtype: int64

=== VALORES ÚNICOS CLAVE ===
ESTADO:       ['RECIBIDOS' 'VIGENTES' 'FINALIZADOS']
ANIO:         ['2,015', '2,016', '2,017', '2,018', '2,019', '2,020', '2,021', '2,022', '2,023', '2.023']
TIPO_ENTIDAD únicos: 79

=== TOP 10 ENTIDADES POR QUEJAS ===
NOMBRE_ENTIDAD
BANCOLOMBIA                      424711
BANCO DE BOGOTA                  344795
BANCO DAVIVIENDA                 273949
BBVA COLOMBIA                    271882
BANCO COLPATRIA, "SCOTIABANK"    158399
AV VILLAS                        148434
BANCO POPULAR                    140714
BANCO DE OCCIDENTE               128950
BANCO FALABELLA S.A.             118610
COLPENSIONES                      98056
Name: count, dtype: int64

=== TOP 10 PRODUCTOS ===
PRODUCTO
TARJETA DE CREDITO                  

In [9]:
#Limpieza completa
df = pd.read_csv(ruta, low_memory=False, encoding='utf-8')
print(f"Registros originales del CSV: {len(df):,}")

# 1. Limpiar ANIO — quitar comas, puntos y convertir a entero
df['ANIO'] = df['ANIO'].astype(str).str.replace(',', '').str.replace('.', '').str.strip()
df['ANIO'] = pd.to_numeric(df['ANIO'], errors='coerce').astype('Int64')

# 2. Eliminar filas con valor "PRUEBA" en PRODUCTO o MOTIVO
df = df[~df['PRODUCTO'].str.upper().str.strip().eq('PRUEBA')]
df = df[~df['MOTIVO'].str.upper().str.strip().eq('PRUEBA')]

# 3. Eliminar filas donde NOMBRE_ENTIDAD es nulo (sin entidad identificada)
df = df[df['NOMBRE_ENTIDAD'].notna()]

# 4. Eliminar filas donde TIPO_ENTIDAD es "?" o nulo
df = df[~df['TIPO_ENTIDAD'].astype(str).str.strip().isin(['?', 'nan', 'NA'])]
df = df[df['TIPO_ENTIDAD'].notna()]

# 5. Limpiar espacios en columnas de texto
for col in ['NOMBRE_ENTIDAD', 'PRODUCTO', 'MOTIVO', 'ESTADO']:
    df[col] = df[col].astype(str).str.strip().str.upper()

# 6. Convertir TIPO_ENTIDAD y CODIGO_ENTIDAD a numérico
df['TIPO_ENTIDAD'] = pd.to_numeric(df['TIPO_ENTIDAD'], errors='coerce').astype('Int64')
df['CODIGO_ENTIDAD'] = pd.to_numeric(df['CODIGO_ENTIDAD'], errors='coerce').astype('Int64')

print(f"Registros después de limpiar: {len(df):,}")
print(f"Registros eliminados: {3754916 - len(df):,}")
print(f"\nNulos restantes:\n{df.isnull().sum()}")
print(f"\nAños únicos: {sorted(df['ANIO'].dropna().unique())}")

Registros originales del CSV: 3,754,916
Registros después de limpiar: 3,531,511
Registros eliminados: 223,405

Nulos restantes:
TIPO_ENTIDAD      0
CODIGO_ENTIDAD    0
NOMBRE_ENTIDAD    0
PRODUCTO          0
MOTIVO            0
ESTADO            0
ANIO              0
MES               0
dtype: int64

Años únicos: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]


In [16]:
#Verificacion final
print("=== MUESTRA LIMPIA ===")
print(df.head(5).to_string())

print(f"\n=== TIPOS DE DATOS FINALES ===")
print(df.dtypes)

print(f"\n=== DISTRIBUCIÓN POR AÑO ===")
print(df.groupby('ANIO').size().reset_index(name='quejas'))

=== MUESTRA LIMPIA ===
   TIPO_ENTIDAD  CODIGO_ENTIDAD                        NOMBRE_ENTIDAD                          PRODUCTO                                                    MOTIVO     ESTADO  ANIO  MES
0             1              57                  BANCO PICHINCHA S.A.  CREDITO DE CONSUMO Y/O COMERCIAL  ASPECTOS CONTRACTUALES (INCUMPLIMIENTO Y/O MODIFICACION)  RECIBIDOS  2016    2
1            23              10  COLFONDOS S.A. PENSIONES Y CESANTIAS                  PENSION DE VEJEZ                    MORA EN EL PAGO O EN EL RECONOCIMIENTO   VIGENTES  2016   11
2             1              49                             AV VILLAS                               NAN                                                       NAN   VIGENTES  2016    9
3             1               7                           BANCOLOMBIA                TARJETA DE CREDITO                              FALLAS EN DATAFONO (COMPRAS)   VIGENTES  2016   12
4             1              39                      BANC

In [17]:
#Reemplazar "NAN" texto por nulo real, luego por etiqueta util
df['PRODUCTO'] = df['PRODUCTO'].replace('NAN', 'SIN ESPECIFICAR')
df['MOTIVO']   = df['MOTIVO'].replace('NAN', 'SIN ESPECIFICAR')

#Excluir 2023 por datos incompletos
df = df[df['ANIO'] != 2023]

print(f"Registros finales (2015-2022): {len(df):,}")
print(f"\nProductos únicos: {df['PRODUCTO'].nunique()}")
print(f"Motivos únicos:   {df['MOTIVO'].nunique()}")
print(f"\nTop 5 productos:\n{df['PRODUCTO'].value_counts().head()}")

Registros finales (2015-2022): 3,527,393

Productos únicos: 124
Motivos únicos:   285

Top 5 productos:
PRODUCTO
TARJETA DE CREDITO                  781848
CREDITO DE CONSUMO Y/O COMERCIAL    781363
CUENTA DE AHORRO                    684070
SIN ESPECIFICAR                     303690
PENSION DE VEJEZ                    136303
Name: count, dtype: int64


In [18]:
#Analisis exploratorio
print("=== QUEJAS POR AÑO ===")
print(df.groupby('ANIO')['NOMBRE_ENTIDAD'].count().reset_index(name='quejas').to_string(index=False))

print("\n=== TOP 10 ENTIDADES ===")
print(df['NOMBRE_ENTIDAD'].value_counts().head(10).to_string())

print("\n=== TOP 10 PRODUCTOS ===")
print(df['PRODUCTO'].value_counts().head(10).to_string())

print("\n=== TOP 10 MOTIVOS ===")
print(df['MOTIVO'].value_counts().head(10).to_string())

print("\n=== QUEJAS POR ESTADO ===")
print(df['ESTADO'].value_counts().to_string())

print("\n=== TIPO ENTIDAD MÁS QUEJADO ===")
print(df['TIPO_ENTIDAD'].value_counts().head(10).to_string())

=== QUEJAS POR AÑO ===
 ANIO  quejas
 2015  375085
 2016  363170
 2017  380360
 2018  422526
 2019  485870
 2020  639650
 2021  594190
 2022  266542

=== TOP 10 ENTIDADES ===
NOMBRE_ENTIDAD
BANCOLOMBIA                      400631
BANCO DE BOGOTA                  334986
BBVA COLOMBIA                    268408
BANCO DAVIVIENDA                 258770
AV VILLAS                        148145
BANCO COLPATRIA, "SCOTIABANK"    142390
BANCO POPULAR                    133638
BANCO DE OCCIDENTE               126619
BANCO FALABELLA S.A.             117596
COLPATRIA RED MULTIBANCA          94344

=== TOP 10 PRODUCTOS ===
PRODUCTO
TARJETA DE CREDITO                  781848
CREDITO DE CONSUMO Y/O COMERCIAL    781363
CUENTA DE AHORRO                    684070
SIN ESPECIFICAR                     303690
PENSION DE VEJEZ                    136303
CREDITO DE VIVIENDA                 124296
OTROS SEGUROS                        63330
AUTOMOVILES                          60422
SEGURO VIDA GRUPO              

In [19]:
# Confirmar qué es TIPO_ENTIDAD = 1
print("=== TIPO_ENTIDAD 1 — muestra de entidades ===")
print(df[df['TIPO_ENTIDAD']==1]['NOMBRE_ENTIDAD'].value_counts().head(10))

# Tasa de resolución por entidad (% FINALIZADOS)
print("\n=== TASA DE RESOLUCIÓN TOP 15 ENTIDADES ===")
top_entidades = df['NOMBRE_ENTIDAD'].value_counts().head(15).index
df_top = df[df['NOMBRE_ENTIDAD'].isin(top_entidades)]

resolucion = df_top.groupby('NOMBRE_ENTIDAD')['ESTADO'].apply(
    lambda x: round((x == 'FINALIZADOS').sum() / len(x) * 100, 1)
).reset_index()
resolucion.columns = ['ENTIDAD', 'PCT_RESUELTO']
resolucion = resolucion.sort_values('PCT_RESUELTO', ascending=False)
print(resolucion.to_string(index=False))

# Evolución anual de los 3 bancos más quejados
print("\n=== BANCOLOMBIA / BOGOTÁ / BBVA POR AÑO ===")
bancos = ['BANCOLOMBIA','BANCO DE BOGOTA','BBVA COLOMBIA']
evol = df[df['NOMBRE_ENTIDAD'].isin(bancos)].groupby(
    ['ANIO','NOMBRE_ENTIDAD']).size().reset_index(name='quejas')
print(evol.to_string(index=False))

# Verificar 2022 — ¿cuántos meses tiene?
print("\n=== MESES DISPONIBLES EN 2022 ===")
print(df[df['ANIO']==2022]['MES'].value_counts().sort_index())

=== TIPO_ENTIDAD 1 — muestra de entidades ===
NOMBRE_ENTIDAD
BANCOLOMBIA                      400631
BANCO DE BOGOTA                  334986
BBVA COLOMBIA                    268408
BANCO DAVIVIENDA                 258770
AV VILLAS                        148145
BANCO COLPATRIA, "SCOTIABANK"    142390
BANCO POPULAR                    133638
BANCO DE OCCIDENTE               126619
BANCO FALABELLA S.A.             117596
COLPATRIA RED MULTIBANCA          94344
Name: count, dtype: int64

=== TASA DE RESOLUCIÓN TOP 15 ENTIDADES ===
                      ENTIDAD  PCT_RESUELTO
    SCOTIABANK COLPATRIA S.A.          32.8
                     PORVENIR          27.0
             BANCO DAVIVIENDA          26.6
              BANCO DE BOGOTA          24.2
                BANCO POPULAR          23.7
                  BANCOLOMBIA          23.7
                BBVA COLOMBIA          23.0
                    AV VILLAS          22.9
BANCO COLPATRIA, "SCOTIABANK"          22.9
           BANCO DE OCCIDENT

In [ ]:
# Excluir 2022 y 2023 del análisis (años incompletos)
df_clean = df[df['ANIO'] <= 2021].copy()
print(f"Dataset final para análisis: {len(df_clean):,} registros (2015-2021)")

# ── TABLA 1: Quejas por año ──────────────────────────────
t1 = df_clean.groupby('ANIO').size().reset_index(name='QUEJAS')
t1.to_csv('t1_quejas_por_anio.csv', index=False)
print(f"✅ t1 exportada: {len(t1)} filas")

# ── TABLA 2: Top entidades con tasa de resolución ────────
t2 = df_clean.groupby('NOMBRE_ENTIDAD').agg(
    QUEJAS=('NOMBRE_ENTIDAD','count'),
    FINALIZADOS=('ESTADO', lambda x: (x=='FINALIZADOS').sum()),
    VIGENTES=('ESTADO', lambda x: (x=='VIGENTES').sum()),
    RECIBIDOS=('ESTADO', lambda x: (x=='RECIBIDOS').sum())
).reset_index()
t2['PCT_RESUELTO'] = round(t2['FINALIZADOS'] / t2['QUEJAS'] * 100, 1)
t2 = t2.sort_values('QUEJAS', ascending=False)
t2.to_csv('t2_entidades.csv', index=False)
print(f"✅ t2 exportada: {len(t2)} filas")

# ── TABLA 3: Quejas por producto ─────────────────────────
t3 = df_clean.groupby('PRODUCTO').size().reset_index(name='QUEJAS')
t3 = t3[t3['PRODUCTO'] != 'SIN ESPECIFICAR']
t3 = t3.sort_values('QUEJAS', ascending=False)
t3.to_csv('t3_productos.csv', index=False)
print(f"✅ t3 exportada: {len(t3)} filas")

# ── TABLA 4: Quejas por motivo ───────────────────────────
t4 = df_clean.groupby('MOTIVO').size().reset_index(name='QUEJAS')
t4 = t4[t4['MOTIVO'] != 'SIN ESPECIFICAR']
t4 = t4.sort_values('QUEJAS', ascending=False)
t4.to_csv('t4_motivos.csv', index=False)
print(f"✅ t4 exportada: {len(t4)} filas")

# ── TABLA 5: Evolución top 5 bancos por año ──────────────
top5 = df_clean['NOMBRE_ENTIDAD'].value_counts().head(5).index.tolist()
t5 = df_clean[df_clean['NOMBRE_ENTIDAD'].isin(top5)].groupby(
    ['ANIO','NOMBRE_ENTIDAD']).size().reset_index(name='QUEJAS')
t5.to_csv('t5_evolucion_top5.csv', index=False)
print(f"✅ t5 exportada: {len(t5)} filas")

# ── TABLA 6: Quejas por mes (estacionalidad) ─────────────
t6 = df_clean.groupby(['ANIO','MES']).size().reset_index(name='QUEJAS')
t6.to_csv('t6_mensual.csv', index=False)
print(f"✅ t6 exportada: {len(t6)} filas")

print("\n✅ Todas las tablas exportadas. Listas para importar en Power BI.")